# Predictive Likelihood Comparison

This notebook summarizes the headline common-evaluation predictive-score comparison. Both filters are updated on the q160 panel and evaluated on the common q300 panel.

ESS and runtime diagnostics are kept in `particle_stability_numerical_behaviour.ipynb`; four-window robustness is kept in `nonoverlapping_window_analysis.ipynb`.

In [ ]:
from pathlib import Path
import math
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
def find_project_root(start=None):
    here = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root containing src/ and notebooks/.")


def project_relative(path):
    path = Path(path)
    try:
        return path.relative_to(PROJECT_ROOT)
    except ValueError:
        return path


PROJECT_ROOT = find_project_root()
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
RUN_ROOT = OUTPUTS_DIR / "common_eval_q160_eval300"

NORMAL_RUN_PATTERN = "normal_400ts_q160_eval300_seed*"
ROUGH_RUN_PATTERN = "rough_400ts_q160_eval300_seed*"
PREDICTIVE_SCORE_FILE = "predictive_loglikelihood_eval.csv"

MAX_TIMESTAMPS = 400
HAC_LAG = 5

COMPARISON_DIR = OUTPUTS_DIR / "comparisons" / "predictive_likelihood"
FIGURE_DIR = COMPARISON_DIR / "figures"
TABLE_DIR = COMPARISON_DIR / "tables"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

## Load Paired Runs

The comparison keeps only seeds and timestamps available for both filters.

In [ ]:
def extract_seed(folder: Path) -> int:
    match = re.search(r"seed(\d+)", folder.name)
    if match is None:
        raise ValueError(f"Seed not found in folder name: {folder.name}")
    return int(match.group(1))


def discover_runs(pattern):
    return {
        extract_seed(path): path
        for path in sorted(RUN_ROOT.glob(pattern))
        if path.is_dir()
    }


normal_runs = discover_runs(NORMAL_RUN_PATTERN)
rough_runs = discover_runs(ROUGH_RUN_PATTERN)
seeds = sorted(set(normal_runs).intersection(rough_runs))
if not seeds:
    raise FileNotFoundError(f"No paired runs found under {project_relative(RUN_ROOT)}")

inventory = pd.DataFrame(
    [
        {
            "seed": seed,
            "normal_run": str(project_relative(normal_runs[seed])),
            "rough_run": str(project_relative(rough_runs[seed])),
            "normal_score_file": (normal_runs[seed] / PREDICTIVE_SCORE_FILE).exists(),
            "rough_score_file": (rough_runs[seed] / PREDICTIVE_SCORE_FILE).exists(),
        }
        for seed in seeds
    ]
)
inventory.to_csv(TABLE_DIR / "predictive_likelihood_run_inventory.csv", index=False)
display(inventory)

In [ ]:
def load_predictive_scores(csv_path: Path, column_name: str, prefix: str) -> pd.DataFrame:
    data = pd.read_csv(csv_path)
    possible_columns = [
        "predictive_log_likelihood",
        "predictive_loglikelihood",
        "predictive_loglik",
        "predictive_log_lik",
        "log_predictive_likelihood",
    ]
    score_column = next((column for column in possible_columns if column in data.columns), None)
    if score_column is None:
        raise ValueError(f"Predictive score column not found in {project_relative(csv_path)}")

    optional_columns = ["capture_time_utc", "n_quotes", "n_update_quotes", "eval_n_options", "eval_n_priced"]
    available = [column for column in optional_columns if column in data.columns]
    result = data[["t_index", score_column, *available]].copy()
    rename = {
        score_column: column_name,
        "capture_time_utc": f"{prefix}_capture_time_utc",
        "n_quotes": f"{prefix}_n_eval_quotes",
        "n_update_quotes": f"{prefix}_n_update_quotes",
        "eval_n_options": f"{prefix}_eval_n_options",
        "eval_n_priced": f"{prefix}_eval_n_priced",
    }
    result = result.rename(columns={old: new for old, new in rename.items() if old in result.columns})
    return (
        result.dropna(subset=["t_index", column_name])
        .sort_values("t_index")
        .drop_duplicates("t_index")
        .reset_index(drop=True)
    )


paired_runs = []
for seed in seeds:
    normal = load_predictive_scores(normal_runs[seed] / PREDICTIVE_SCORE_FILE, "normal_predictive_score", "normal")
    rough = load_predictive_scores(rough_runs[seed] / PREDICTIVE_SCORE_FILE, "rough_predictive_score", "rough")
    paired_seed = normal.merge(rough, on="t_index", how="inner", validate="one_to_one")

    if {"normal_capture_time_utc", "rough_capture_time_utc"}.issubset(paired_seed.columns):
        if not (paired_seed["normal_capture_time_utc"] == paired_seed["rough_capture_time_utc"]).all():
            raise ValueError(f"Capture times differ between models for seed {seed}.")

    paired_seed["seed"] = seed
    paired_runs.append(paired_seed)

common_timestamps = sorted(set.intersection(*[set(frame["t_index"]) for frame in paired_runs]))
if MAX_TIMESTAMPS is not None:
    common_timestamps = common_timestamps[:MAX_TIMESTAMPS]

paired = pd.concat(paired_runs, ignore_index=True)
paired = paired[paired["t_index"].isin(common_timestamps)].sort_values(["seed", "t_index"]).reset_index(drop=True)
observations_per_seed = paired.groupby("seed")["t_index"].nunique()
if observations_per_seed.nunique() != 1:
    raise ValueError("The paired runs do not contain the same timestamp count for every seed.")

paired["one_step_difference"] = paired["rough_predictive_score"] - paired["normal_predictive_score"]
paired["normal_cumulative_score"] = paired.groupby("seed")["normal_predictive_score"].cumsum()
paired["rough_cumulative_score"] = paired.groupby("seed")["rough_predictive_score"].cumsum()
paired["cumulative_difference"] = paired["rough_cumulative_score"] - paired["normal_cumulative_score"]

paired.to_csv(TABLE_DIR / "predictive_likelihood_paired_scores.csv", index=False)
print(f"Comparison uses {len(seeds)} seeds and {int(observations_per_seed.iloc[0])} timestamps per seed.")
display(paired.head())

## Quote-Set Audit

The paired score comparison is meaningful only when both filters use matching update and evaluation panels.

In [ ]:
quote_summary = (
    paired.groupby("seed", as_index=False)
    .agg(
        timestamps=("t_index", "nunique"),
        normal_eval_quotes=("normal_n_eval_quotes", "sum"),
        rough_eval_quotes=("rough_n_eval_quotes", "sum"),
        mean_eval_quotes_per_timestamp=("normal_n_eval_quotes", "mean"),
        min_eval_quotes_per_timestamp=("normal_n_eval_quotes", "min"),
        max_eval_quotes_per_timestamp=("normal_n_eval_quotes", "max"),
        normal_update_quotes=("normal_n_update_quotes", "sum"),
        rough_update_quotes=("rough_n_update_quotes", "sum"),
        mean_update_quotes_per_timestamp=("normal_n_update_quotes", "mean"),
        normal_eval_options=("normal_eval_n_options", "sum"),
        normal_eval_priced=("normal_eval_n_priced", "sum"),
        rough_eval_options=("rough_eval_n_options", "sum"),
        rough_eval_priced=("rough_eval_n_priced", "sum"),
    )
)
quote_summary["normal_invalid_eval_pricings"] = quote_summary["normal_eval_options"] - quote_summary["normal_eval_priced"]
quote_summary["rough_invalid_eval_pricings"] = quote_summary["rough_eval_options"] - quote_summary["rough_eval_priced"]

if not (quote_summary["normal_eval_quotes"] == quote_summary["rough_eval_quotes"]).all():
    raise ValueError("Normal and Rough are not evaluated on the same number of quotes.")
if not (quote_summary["normal_update_quotes"] == quote_summary["rough_update_quotes"]).all():
    raise ValueError("Normal and Rough are not updated on the same number of quotes.")

quote_summary.to_csv(TABLE_DIR / "q160_update_q300_evaluation_quote_check.csv", index=False)
display(quote_summary)

## Headline Score Summary

Positive Rough-minus-Normal values favor the rough filter.

In [ ]:
seed_summary = (
    paired.groupby("seed", as_index=False)
    .agg(
        timestamps=("t_index", "nunique"),
        mean_one_step_difference=("one_step_difference", "mean"),
        median_one_step_difference=("one_step_difference", "median"),
        final_normal_cumulative_score=("normal_cumulative_score", "last"),
        final_rough_cumulative_score=("rough_cumulative_score", "last"),
        final_cumulative_difference=("cumulative_difference", "last"),
    )
)
seed_summary["preferred_model"] = np.where(seed_summary["final_cumulative_difference"] > 0, "Rough-SABR", "Normal SABR")
seed_summary.to_csv(TABLE_DIR / "predictive_likelihood_seed_summary.csv", index=False)

across_seeds = (
    paired.groupby("t_index", as_index=False)
    .agg(
        normal_one_step_mean=("normal_predictive_score", "mean"),
        normal_one_step_std=("normal_predictive_score", "std"),
        rough_one_step_mean=("rough_predictive_score", "mean"),
        rough_one_step_std=("rough_predictive_score", "std"),
        normal_cumulative_mean=("normal_cumulative_score", "mean"),
        normal_cumulative_std=("normal_cumulative_score", "std"),
        rough_cumulative_mean=("rough_cumulative_score", "mean"),
        rough_cumulative_std=("rough_cumulative_score", "std"),
        one_step_difference_mean=("one_step_difference", "mean"),
        one_step_difference_std=("one_step_difference", "std"),
        cumulative_difference_mean=("cumulative_difference", "mean"),
        cumulative_difference_std=("cumulative_difference", "std"),
    )
)
across_seeds.to_csv(TABLE_DIR / "predictive_likelihood_timestamp_summary.csv", index=False)
display(seed_summary.round(4))

In [ ]:
def newey_west_mean_inference(values, lag=5):
    x = pd.Series(values).dropna().to_numpy(dtype=float)
    n = len(x)
    if n < 3:
        raise ValueError("At least three timestamp observations are needed for HAC inference.")

    selected_lag = max(0, min(int(lag), n - 1))
    mean = float(np.mean(x))
    centered = x - mean
    gamma0 = float(np.dot(centered, centered) / n)
    long_run_variance = gamma0
    for ell in range(1, selected_lag + 1):
        autocov = float(np.dot(centered[ell:], centered[:-ell]) / n)
        weight = 1.0 - ell / (selected_lag + 1.0)
        long_run_variance += 2.0 * weight * autocov

    long_run_variance = max(long_run_variance, 0.0)
    standard_error = math.sqrt(long_run_variance / n)
    t_statistic = mean / standard_error if standard_error > 0 else np.nan
    p_value_one_sided = 0.5 * math.erfc(t_statistic / math.sqrt(2.0)) if np.isfinite(t_statistic) else np.nan
    return {
        "n_timestamps": n,
        "hac_lag": selected_lag,
        "mean_difference": mean,
        "median_difference": float(np.median(x)),
        "cumulative_difference": float(np.sum(x)),
        "hac_standard_error": standard_error,
        "hac_t_statistic": t_statistic,
        "hac_p_value_one_sided": p_value_one_sided,
        "hac_ci_lower_95": mean - 1.96 * standard_error,
        "hac_ci_upper_95": mean + 1.96 * standard_error,
    }


diff_series = across_seeds["one_step_difference_mean"]
hac_result = newey_west_mean_inference(diff_series, lag=HAC_LAG)
rough_wins = int((diff_series > 0).sum())

hac_numeric = pd.DataFrame(
    [
        {
            **hac_result,
            "seeds_averaged_per_timestamp": len(seeds),
            "rough_win_count": rough_wins,
            "rough_win_fraction": rough_wins / hac_result["n_timestamps"],
        }
    ]
)
hac_numeric.to_csv(TABLE_DIR / "timestamp_hac_inference_numeric.csv", index=False)

hac_display = pd.DataFrame(
    {
        "Quantity": [
            "Timestamps",
            "Seeds averaged per timestamp",
            "Newey-West lag",
            "Mean Rough - Normal score",
            "Median Rough - Normal score",
            "Cumulative Rough - Normal score",
            "Rough wins by timestamp",
            "HAC standard error",
            "HAC 95% confidence interval",
            "One-sided HAC p-value",
        ],
        "Value": [
            f"{hac_result['n_timestamps']}",
            f"{len(seeds)}",
            f"{hac_result['hac_lag']}",
            f"{hac_result['mean_difference']:.4f}",
            f"{hac_result['median_difference']:.4f}",
            f"{hac_result['cumulative_difference']:.2f}",
            f"{rough_wins} / {hac_result['n_timestamps']} ({100 * rough_wins / hac_result['n_timestamps']:.2f}%)",
            f"{hac_result['hac_standard_error']:.4f}",
            f"[{hac_result['hac_ci_lower_95']:.4f}, {hac_result['hac_ci_upper_95']:.4f}]",
            "<0.001" if hac_result["hac_p_value_one_sided"] < 0.001 else f"{hac_result['hac_p_value_one_sided']:.4f}",
        ],
    }
)
hac_display.to_csv(TABLE_DIR / "timestamp_hac_inference_display.csv", index=False)
display(hac_display)

## Score Paths

The figure shows the seed-averaged one-step score difference and cumulative score difference.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
x = across_seeds["t_index"].to_numpy(dtype=float)

y = across_seeds["one_step_difference_mean"].to_numpy(dtype=float)
s = across_seeds["one_step_difference_std"].fillna(0.0).to_numpy(dtype=float)
axes[0].plot(x, y, color="#1f77b4", linewidth=1.5)
axes[0].fill_between(x, y - s, y + s, color="#1f77b4", alpha=0.16, linewidth=0)
axes[0].axhline(0.0, color="black", linestyle="--", linewidth=0.9)
axes[0].set_title("One-step predictive score")
axes[0].set_xlabel("Timestamp index")
axes[0].set_ylabel("Rough - Normal")
axes[0].grid(alpha=0.25)

y = across_seeds["cumulative_difference_mean"].to_numpy(dtype=float)
s = across_seeds["cumulative_difference_std"].fillna(0.0).to_numpy(dtype=float)
axes[1].plot(x, y, color="#d65225", linewidth=1.5)
axes[1].fill_between(x, y - s, y + s, color="#d65225", alpha=0.16, linewidth=0)
axes[1].axhline(0.0, color="black", linestyle="--", linewidth=0.9)
axes[1].set_title("Cumulative predictive score")
axes[1].set_xlabel("Timestamp index")
axes[1].set_ylabel("Rough - Normal")
axes[1].grid(alpha=0.25)

fig.tight_layout()
fig.savefig(FIGURE_DIR / "predictive_likelihood_score_differences.png", dpi=220, bbox_inches="tight")
plt.show()

## Four-Window HAC Lag Check

This optional check reads the four-window timestamp differences produced by `nonoverlapping_window_analysis.ipynb`.

In [ ]:
NONOVERLAP_TABLE_DIR = OUTPUTS_DIR / "nonoverlap_q160_eval300" / "_comparison" / "tables"
lag_input_path = NONOVERLAP_TABLE_DIR / "nonoverlap_seed_averaged_timestamp_deltas.csv"

if lag_input_path.exists():
    lag_sensitivity_input = pd.read_csv(lag_input_path)
    hac_lags_to_check = [3, 5, 10, 20]
    window_order = ["w0_400", "w401_800", "w801_1200", "w1201_1600"]
    window_labels = {
        "w0_400": "0-399",
        "w401_800": "400-799",
        "w801_1200": "800-1199",
        "w1201_1600": "1200-1599",
    }

    rows = []
    for window in window_order:
        window_data = (
            lag_sensitivity_input.loc[
                lag_sensitivity_input["window"] == window,
                ["t_index", "delta_log_score"],
            ]
            .sort_values("t_index")
            .reset_index(drop=True)
        )
        for lag in hac_lags_to_check:
            inference = newey_west_mean_inference(window_data["delta_log_score"], lag=lag)
            rows.append(
                {
                    "Window": window_labels[window],
                    "HAC lag": lag,
                    "Timestamps": inference["n_timestamps"],
                    "Mean Rough - Normal": inference["mean_difference"],
                    "HAC SE": inference["hac_standard_error"],
                    "HAC 95% CI lower": inference["hac_ci_lower_95"],
                    "HAC 95% CI upper": inference["hac_ci_upper_95"],
                    "One-sided HAC p-value": inference["hac_p_value_one_sided"],
                }
            )

    hac_lag_sensitivity = pd.DataFrame(rows)
    hac_lag_sensitivity.to_csv(NONOVERLAP_TABLE_DIR / "nonoverlap_hac_lag_sensitivity.csv", index=False)
    display(hac_lag_sensitivity.round(4))
else:
    print(f"Skipped: {project_relative(lag_input_path)} does not exist yet.")